<a href="https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula4-exercicio3-pesos-da-matriz-cheia.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 4, exercício 3: Pesos da matriz cheia

Quantos números tem a matriz `W`, a que a gente não mexe?

O exercício é da aula [4. Medir no seu dado](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/notebooks/04-medir-no-seu-dado.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara o modelo, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [ ]:
!pip install -q torch torchvision

## Preparação

A preparação monta o par `A` e `B` ao lado da matriz `W`, extrai o vetor da base e ajusta uma cabeça linear. A prova e o conjunto de cor trocada ficam de fora desse ajuste.

In [ ]:
def contar(modelo, so_treinavel=False):
    total = 0
    for p in modelo.parameters():
        if so_treinavel and not p.requires_grad:
            continue
        total += p.numel()
    return total


def lote_formas(n, semente, trocar_cor=False):
    """Desenha círculos e quadrados. Classe 0 é círculo, classe 1 é quadrado."""
    g = torch.Generator().manual_seed(semente)
    y = torch.randint(0, 2, (n,), generator=g)
    x = torch.full((n, 3, 64, 64), 0.08)
    yy, xx = torch.meshgrid(torch.arange(64), torch.arange(64), indexing="ij")
    circulo = (yy - 32) ** 2 + (xx - 32) ** 2 <= 16 ** 2
    for i in range(n):
        cor = torch.tensor([0.90, 0.20, 0.15] if int(y[i]) == 0 else [0.15, 0.35, 0.92])
        if trocar_cor:
            cor = cor.flip(0)
        if int(y[i]) == 0:
            x[i, :, circulo] = cor.view(3, 1)
        else:
            x[i, :, 16:48, 16:48] = cor.view(3, 1, 1)
    return x, y


def normalizar(x):
    media = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    desvio = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (x - media) / desvio

import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(7)
posto = 4
entrada = 512
A = torch.zeros(posto, entrada)
B = torch.zeros(entrada, posto)
W = torch.zeros(entrada, entrada)
params_par = A.numel() + B.numel()
params_cheia = W.numel()

x_treino, y_treino = lote_formas(32, semente=11)
x_prova, y_prova = lote_formas(16, semente=29)
x_longe, y_longe = lote_formas(16, semente=29, trocar_cor=True)
x_treino, x_prova, x_longe = normalizar(x_treino), normalizar(x_prova), normalizar(x_longe)

base = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
base.fc = nn.Identity()
base.eval()


def extrair(x):
    with torch.no_grad():
        return base(x)


feats_treino = extrair(x_treino)
feats_prova = extrair(x_prova)
feats_longe = extrair(x_longe)

peso = torch.zeros(2, feats_treino.shape[1], requires_grad=True)
vies = torch.zeros(2, requires_grad=True)
opt = torch.optim.Adam([peso, vies], lr=1e-2)
for _ in range(40):
    opt.zero_grad()
    perda = nn.functional.cross_entropy(feats_treino @ peso.T + vies, y_treino)
    perda.backward()
    opt.step()
peso, vies = peso.detach(), vies.detach()


def acuracia_linear(peso, vies, feats, y):
    with torch.no_grad():
        pred = (feats @ peso.T + vies).argmax(1)
    return float((pred == y).float().mean())


acerto_seu = acuracia_linear(peso, vies, feats_prova, y_prova)
acerto_longe = acuracia_linear(peso, vies, feats_longe, y_longe)
print(f"posto {posto} | par {params_par} pesos | matriz cheia {params_cheia} pesos")
print(f"prova: {len(y_prova)} fotos | acerto no seu dado {acerto_seu:.0%} | acerto com a cor trocada {acerto_longe:.0%}")
print(f"o par cabe na matriz? {tuple((B @ A).shape) == tuple(W.shape)}")

In [ ]:
# @title Correção automática (rode, não precisa ler)
VERIFICAR = {
    1: lambda r: int(r) == A.shape[0],
    2: lambda r: int(r) == A.numel() + B.numel(),
    3: lambda r: int(r) == W.numel(),
    4: lambda r: int(r) == len(y_prova),
    5: lambda r: abs(float(r) - acuracia_linear(peso, vies, feats_prova, y_prova)) < 1e-5,
    6: lambda r: abs(float(r) - acuracia_linear(peso, vies, feats_longe, y_longe)) < 1e-5,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Quantos números tem a matriz `W`, a que a gente não mexe?

*Dica:* W.numel()

Troque o `None` pela sua resposta e rode a célula.

In [ ]:
resposta = None

conferir(3, resposta)

## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [ ]:
resposta = W.numel()

conferir(3, resposta)

## O que fica

Próximo: [Aula 4, exercício 4: Tamanho da prova](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula4-exercicio4-tamanho-da-prova.ipynb).